In [3]:
print("Check_Python")

Check_Python


In [2]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [4]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [7]:
PROJECT_ROOT = Path("..") 
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

TRAIN_DIR, VAL_DIR, TEST_DIR

(WindowsPath('../data/raw/train'),
 WindowsPath('../data/raw/val'),
 WindowsPath('../data/raw/test'))

In [9]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"

In [10]:
#lighter augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

We aren't using advanced data augmentation because it will risk the data. problems:-
- large data
- brightness and contrast matters

In [11]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

In [12]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary"
)

Found 5216 images belonging to 2 classes.


In [13]:
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary"
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False
)

Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [15]:
#Handling class imbalance
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_generator.classes)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes
)

class_weights = dict(zip(classes, class_weights))
class_weights

{np.int32(0): np.float64(1.9448173005219984),
 np.int32(1): np.float64(0.6730322580645162)}

The solution we used: Class Weights
* If a class has fewer images → its mistakes are more costly
* If a class has many images → its mistakes cost less